# Figure S1D — Pre-RAG vs Post-RAG Cost Reduction (55,406-Patient Extrapolation)

Bar chart showing estimated LLM inference cost reduction from RAG-based input compression.

**Method:**
- Anchor: actual post-RAG Bedrock cost for 70K patients = $2,904.77

- Character ratio from 110-patient dataset scales pre-RAG cost

- RAG infrastructure (SageMaker): $558.50 for 70K, scaled proportionally

**Data sources** (under `figures/figures_data/figure 1/data`):
- `comprehensive_evaluation.csv` — pre-RAG 110-patient notes (character counts)
- `enhanced_rag_dataset.csv` — post-RAG 110-patient notes (character counts)

Outputs are written to `figure 1/results/supp/`.


In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

%matplotlib inline

# ---- Hard-fail if Arial isn't actually resolved (no silent fallback) ----
import matplotlib.font_manager as fm
_arial_path = fm.findfont('Arial', fallback_to_default=False)
if 'Arial' not in _arial_path:
    raise RuntimeError(
        f"Arial not found -- matplotlib resolved to '{_arial_path}' instead. "
        "Install Arial or update font.sans-serif before rendering this figure."
    )
print(f"Arial resolved to: {_arial_path}")


In [ ]:
# ---------------------------------------------------------------------------
# FILE PATHS — notebook lives in figure 1/scripts/
#   figures/
#   ├── figures_data/figure 1/data/   ← inputs (shared OneDrive data dir)
#   └── v1/figure 1/
#       ├── scripts/                  ← this notebook
#       └── results/supp/             ← output PDF + CSV
# ---------------------------------------------------------------------------
ROOT = Path("..").resolve()
FIGURES = ROOT.parent.parent
DATA = FIGURES / "figures_data" / "figure 1" / "data"
RESULTS = ROOT / "results"
(RESULTS / "supp").mkdir(parents=True, exist_ok=True)

PRE_RAG_CSV  = DATA / "comprehensive_evaluation.csv"
POST_RAG_CSV = DATA / "enhanced_rag_dataset.csv"
OUT_PATH     = RESULTS / "supp" / "Pre_vs_Post_RAG_Cost_S1D.pdf"

for p in [PRE_RAG_CSV, POST_RAG_CSV]:
    assert p.exists(), f"Missing: {p}"
print("All input files found.")
print(f"Data: {DATA}")
print(f"Results: {RESULTS / 'supp'}")


In [ ]:
# ---------------------------------------------------------------------------
# CONSTANTS
# ---------------------------------------------------------------------------
COLOR_BEFORE = "#E8955A"
COLOR_AFTER  = "#4878CF"

N = 55_406                          # patients to extrapolate to
ACTUAL_POST_RAG_COST_70K = 2904.77  # actual Bedrock bill for 70K patients
SAGEMAKER_RAG_COST_70K   = 558.50   # RAG infra cost for 70K patients
PATIENTS_70K = 70_000


In [ ]:
# ---------------------------------------------------------------------------
# Compute character ratio from 110-patient dataset
# ---------------------------------------------------------------------------
pre_110 = pd.read_csv(PRE_RAG_CSV, low_memory=False)
pre_110 = pre_110[["mrn", "cdd_doc_guid", "cdd_doc_name", "date", "input_text"]].dropna().drop_duplicates()
pre_110_cpp = pre_110["input_text"].astype(str).str.len().sum() / pre_110["mrn"].nunique()

post_110 = pd.read_csv(POST_RAG_CSV, low_memory=False)
post_110_cpp = post_110["batch_text"].astype(str).str.len().sum() / post_110["mrn"].nunique()

char_ratio = pre_110_cpp / post_110_cpp

print(f"Pre-RAG chars/patient:  {pre_110_cpp:,.0f}")
print(f"Post-RAG chars/patient: {post_110_cpp:,.0f}")
print(f"Character ratio:        {char_ratio:.1f}x")


In [ ]:
# ---------------------------------------------------------------------------
# Cost derivation
# ---------------------------------------------------------------------------
scale = N / PATIENTS_70K
post_rag_llm_cost = ACTUAL_POST_RAG_COST_70K * scale
pre_rag_llm_cost  = post_rag_llm_cost * char_ratio
sagemaker_cost    = SAGEMAKER_RAG_COST_70K * scale
post_rag_total    = post_rag_llm_cost + sagemaker_cost

savings   = pre_rag_llm_cost - post_rag_total
pct_saved = (1 - post_rag_total / pre_rag_llm_cost) * 100

print(f"Cost extrapolation to {N:,} patients:")
print(f"  Pre-RAG LLM:           ${pre_rag_llm_cost:>10,.2f}")
print(f"  Post-RAG LLM:          ${post_rag_llm_cost:>10,.2f}")
print(f"  RAG infra (SageMaker):  ${sagemaker_cost:>9,.2f}")
print(f"  Post-RAG total:        ${post_rag_total:>10,.2f}")
print(f"  Savings:               ${savings:>10,.2f} ({pct_saved:.1f}%)")


In [ ]:
# ---------------------------------------------------------------------------
# Figure
# ---------------------------------------------------------------------------
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 6,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig, ax = plt.subplots(figsize=(3.5, 2.0))

pre_cost_k  = pre_rag_llm_cost / 1e3
post_cost_k = post_rag_total / 1e3

bars = ax.bar(
    ["Pre-RAG", "Post-RAG"],
    [pre_cost_k, post_cost_k],
    color=[COLOR_BEFORE, COLOR_AFTER],
    width=0.45, edgecolor="white", linewidth=0.5,
)

ax.set_ylabel("Estimated Cost ($ Thousands)")
ax.set_title("Cost Reduction")
ax.ticklabel_format(axis="y", style="plain", useOffset=False)

for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
for spine in ("bottom", "left"):
    ax.spines[spine].set_linewidth(0.6)
ax.tick_params(width=0.6, length=3)
ax.set_ylim(bottom=0)
ymax = ax.get_ylim()[1]
ax.set_ylim(top=ymax * 1.15)

ax.annotate(f"${pre_rag_llm_cost:,.0f}",
    xy=(bars[0].get_x() + bars[0].get_width() / 2, pre_cost_k),
    ha="center", va="bottom", fontsize=5)
ax.annotate(f"${post_rag_total:,.0f}",
    xy=(bars[1].get_x() + bars[1].get_width() / 2, post_cost_k),
    ha="center", va="bottom", fontsize=5)

fig.tight_layout()
fig.savefig(OUT_PATH, format="pdf", dpi=450)
print(f"Saved: {OUT_PATH}")
plt.show()

In [ ]:
# ===========================================================================
# EXPORT — underlying data for this panel
# ===========================================================================

CSV_OUT = OUT_PATH.with_name(OUT_PATH.stem + "_results.csv")

export_df = pd.DataFrame([
    {"quantity": "pre_rag_chars_per_patient",  "value": pre_110_cpp,           "units": "characters"},
    {"quantity": "post_rag_chars_per_patient", "value": post_110_cpp,          "units": "characters"},
    {"quantity": "character_ratio",            "value": char_ratio,            "units": "ratio"},
    {"quantity": "pre_rag_llm_cost",           "value": pre_rag_llm_cost,      "units": "USD"},
    {"quantity": "post_rag_llm_cost",          "value": post_rag_llm_cost,     "units": "USD"},
    {"quantity": "rag_infra_cost_sagemaker",   "value": sagemaker_cost,        "units": "USD"},
    {"quantity": "post_rag_total_cost",        "value": post_rag_total,        "units": "USD"},
    {"quantity": "absolute_savings",           "value": savings,               "units": "USD"},
    {"quantity": "percent_saved",              "value": pct_saved,             "units": "percent"},
    # anchors / assumptions
    {"quantity": "n_patients_extrapolated",    "value": N,                          "units": "patients"},
    {"quantity": "anchor_post_rag_cost_70k",   "value": ACTUAL_POST_RAG_COST_70K,   "units": "USD"},
    {"quantity": "anchor_sagemaker_cost_70k",  "value": SAGEMAKER_RAG_COST_70K,     "units": "USD"},
    {"quantity": "anchor_n_patients",          "value": PATIENTS_70K,               "units": "patients"},
    {"quantity": "scale_factor",               "value": scale,                      "units": "ratio"},
    {"quantity": "n_patients_char_ratio_basis","value": pre_110["mrn"].nunique(),   "units": "patients"},
])

export_df.to_csv(CSV_OUT, index=False)
print(f"Saved: {CSV_OUT.name}")
print(f"  {len(export_df)} quantities")